# Assignment — regression, model selection, regularization

Solution cells use `### BEGIN/END SOLUTION` (nbgrader). Builds on `notebooks/ddm_regression_basics.ipynb`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..','..')))
import numpy as np, torch, matplotlib.pyplot as plt
from pyutils import pinn_core as pc
from pyutils.pinn_core import MLP
torch.set_default_dtype(torch.float64); torch.manual_seed(0); np.random.seed(0)
pc.set_plot_style()
f=lambda x: np.sin(2*np.pi*x)+0.3*x
n=60; x=np.random.rand(n); y=f(x)+0.1*np.random.randn(n)
# i.i.d. random split (NOT sorted) -> train/val/test cover the same range
perm=np.random.permutation(n); itr,iva,ite=perm[:36],perm[36:48],perm[48:]
xtr,ytr=x[itr],y[itr]; xva,yva=x[iva],y[iva]; xte,yte=x[ite],y[ite]
print(f'{len(xtr)} train / {len(xva)} val / {len(xte)} test')

36 train / 12 val / 12 test


## Task 1 — pick the polynomial degree by validation

Fit polynomials of degree 1..15 on the **training** set, select the degree with the lowest **validation** MSE, and report the **test** MSE (must be below 0.05).

In [2]:
best_deg=None; test_mse=None
### BEGIN SOLUTION
def fit(d,xq):
    w,_,_,_=np.linalg.lstsq(np.vander(xtr,d+1),ytr,rcond=None); return np.vander(xq,d+1)@w
vmse=[np.mean((fit(d,xva)-yva)**2) for d in range(1,16)]
best_deg=int(np.argmin(vmse)+1)
test_mse=float(np.mean((fit(best_deg,xte)-yte)**2))
### END SOLUTION
print(f'best degree={best_deg}, test MSE={test_mse:.4f}')
assert best_deg is not None and test_mse < 0.05

best degree=3, test MSE=0.0144


## Task 2 — regularization in an overfitting regime

With only a **few noisy points** and a very flexible network, the unregularized MLP fits the noise. Train it without and with L2 weight decay and show the regularized model is closer to the **true** function (MSE measured against $f$ on a dense grid).

In [3]:
np.random.seed(1); xs=np.random.rand(12); ys=f(xs)+0.2*np.random.randn(12)
Xs=torch.tensor(xs).reshape(-1,1); Ys=torch.tensor(ys).reshape(-1,1)
xfine=np.linspace(0,1,200); Xf=torch.tensor(xfine).reshape(-1,1); ytruth=f(xfine)
mse_noreg=mse_reg=None
### BEGIN SOLUTION
def fit_mlp(wd):
    torch.manual_seed(0); net=MLP([1,128,128,1]); opt=torch.optim.Adam(net.parameters(),lr=5e-3,weight_decay=wd)
    for _ in range(6000):
        opt.zero_grad(); l=((net(Xs)-Ys)**2).mean(); l.backward(); opt.step()
    with torch.no_grad(): return float(((net(Xf).numpy().ravel()-ytruth)**2).mean())
mse_noreg=fit_mlp(0.0); mse_reg=fit_mlp(1e-2)
### END SOLUTION
print(f'fit-to-truth MSE  no-reg={mse_noreg:.4f}  L2={mse_reg:.4f}')
assert mse_reg < mse_noreg

fit-to-truth MSE  no-reg=0.5405  L2=0.1019
